In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

presto_port = '80'
username = 'airflow-user'
# username = 'nihal.r@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


In [2]:
def consideration_query(run_date,city_name,base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as
    (
    select
    customer_id as customer,
    date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
    ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
    taxi_lifetime_last_ride_date,
    taxi_lifetime_first_ride_date,
    taxi_lifetime_stage,
    previous_taxi_lifetime_stage,
    taxi_lifetime_rides,
    taxi_recency_segment,
    taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
    taxi_lifetime_last_ride_city city_name,
    fe_mean_intent_type,
    fe_intent_trend_type,
    fe_potential_per_type,
    fe_regularity_type,
    fe_recency_type,
    fe_volatility_type,
    gender_tag,
    taxi_income_segment,
    ps_tag_link,
    ps_tag_auto,
    taxi_service_affinity,
    taxi_distance_preference,
    taxi_time_of_day_preference,
    taxi_day_of_week_preference
    from datasets.iallocator_customer_segments
    where date(run_date) = date('{base_week}')
    -- and taxi_lifetime_last_ride_date = date('2023-11-01')
    and taxi_recency_segment in ('RECENT')
    and taxi_lifetime_rides > 4
    and taxi_lifetime_last_ride_city in ('{city_name}')
    )
    ,
    base as 
(
select a.*,
case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,

case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,

case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,

case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10


from dormant_customer a
left join
(
select customerid,
-- sum(ao_sessions_unique_daily) ao,
-- sum(fe_sessions_unique_daily) fe,
-- sum(rr_sessions_unique_daily) rr,
-- sum(net_rides_daily) nr,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,

sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,

sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,


sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

from datasets.customer_rf_daily_kpi 
where date(day)>date('{base_week}') and date(day)<=date('{base_week}')+interval '70' day
and service_name in ('Link', 'Auto', 'CabEconomy')
and city in ('{city_name}')
group by 1
)b
on a.customer=b.customerid

)
    ,
rr_data as(
select fe.*
    from
    (select aa.*, bb.rr_LT, bb.rapido_age,bb.rapido_age_seg
    from
    (select customerid,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
    sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
    sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
    sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
    sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
    sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
    sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
    sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
    sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
    sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
    sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
    sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
    sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
    count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
    count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
    count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
    count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
    count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
    count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
    count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
    count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
    count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
    count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
    count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
    count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
    count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
    day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
      from 
        (select customerid,day, max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
        from 
        datasets.customer_rf_daily_kpi a
        join
        base b
        on a.customerid=b.customer
        and 1=1 
        and city = '{city_name}'
        and rr_sessions_unique_daily >= 1 
        and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
        and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
        and service_name in ('Link','Auto', 'CabEconomy')
        group by customerid, day
        )
      group by 
        customerid) aa 
    left join 
    (select customerid, sum(rr_sessions_unique_daily) as rr_LT , day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
    ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
            from 
        datasets.customer_rf_daily_kpi a
        join
        base b
        on a.customerid=b.customer
        and rr_sessions_unique_daily >= 1 
        and city = '{city_name}'
        group by 1
        ) bb
        on aa.customerid = bb.customerid) fe 
  )
    select a.*,b.*,'{run_date}' as run_date
    from 
    base a
    left join
    rr_data b
    on a.customer=b.customerid


    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    print(data.shape)   
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11',
                       'active_rr_days_w10', 'active_rr_days_w09', 'active_rr_days_w08',
                       'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05',
                       'active_rr_days_w04', 'active_rr_days_w03', 'active_rr_days_w02',
                       'active_rr_days_w01']

    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    
    data['rr_norm_w13']=data.rr_w13/data.active_rr_days_w13

    data['rr_norm_w12']=data.rr_w12/data.active_rr_days_w12

    data['rr_norm_w11']=data.rr_w11/data.active_rr_days_w11

    data['rr_norm_w10']=data.rr_w10/data.active_rr_days_w10

    data['rr_norm_w09']=data.rr_w09/data.active_rr_days_w09

    data['rr_norm_w08']=data.rr_w08/data.active_rr_days_w08

    data['rr_norm_w07']=data.rr_w07/data.active_rr_days_w07

    data['rr_norm_w06']=data.rr_w06/data.active_rr_days_w06

    data['rr_norm_w05']=data.rr_w05/data.active_rr_days_w05

    data['rr_norm_w04']=data.rr_w04/data.active_rr_days_w04

    data['rr_norm_w03']=data.rr_w03/data.active_rr_days_w03

    data['rr_norm_w02']=data.rr_w02/data.active_rr_days_w02

    data['rr_norm_w01']=data.rr_w01/data.active_rr_days_w01

    raw_customer_data=data[data['customerid']!=0]
    back_up = raw_customer_data.copy()
    fe_data2 = raw_customer_data.fillna(0)
    fe_data=fe_data2[fe_data2['customerid']!=0]
    
    def regularity_decider(x,active_mean):
        if x<=0.3:
            return '05_rare_need'
        elif x>0.3 and x<=0.5:
            return '04_monthly'
        elif x>0.5 and x<=0.75:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'
    #         return 'weekly'

    def regularity_decider_v2(x,active_mean):
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'

    def intent_decider(x):
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    


    def consideration_signal_creator(x,y,active_mean):
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
    #             if active_mean>2.6:
    #                 return 'daily'
    #             else:
    #                 return 'weekly'



    def correction_factor(rr_weight,rapido_age):
        if rapido_age >= 91:
            multiplier = 1
            week_inactive = 0

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))        
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        return(multiplier,week_inactive,(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)) , ((rr_weight*rr_weight).sum()- (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6)))

    def recency_type(days_since_last_fe):
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    def intent_trend_type(intent_trend):
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')

#     def custom_func(mean_intent,rr_trend):
#         if mean_intent=='High':
#             return rr_trend
#         elif mean_intent=='Low' and rr_trend=='01_Increasing':
#             return '01_Increasing'
#         else:
#             return '02_Stable'

    def create_L3_signals(fe_data,back_up): 
        
        rr_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']  
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12',
           'active_rr_days_w11', 'active_rr_days_w10', 'active_rr_days_w09',
           'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03',
           'active_rr_days_w02', 'active_rr_days_w01'] 
        norm_intent=['rr_norm_w13', 'rr_norm_w12', 'rr_norm_w11', 'rr_norm_w10',
           'rr_norm_w09', 'rr_norm_w08', 'rr_norm_w07', 'rr_norm_w06',
           'rr_norm_w05', 'rr_norm_w04', 'rr_norm_w03', 'rr_norm_w02',
           'rr_norm_w01']
        fe_data['fe2rr_91days'] = fe_data.apply(lambda x: (x['rr_91days']/x['fe_91days']) if x['fe_91days'] > 0 else 0.5,axis=1)
        
        print('cp1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        fe_data['rr_need'] = np.array(fe_data[fe_cols]).max(axis=1)
        fe_data['no_week_rr_done'] =  back_up[fe_cols].count(axis=1)
        fe_data['%week_rr_done'] = fe_data.apply(lambda x: x['no_week_rr_done']*(1/(13 - correction_factor(rr_weight,x['rapido_age'])[1])), axis = 1 )
#         fe_data['rr_cov'] =  (np.sqrt(back_up[fe_cols].var(axis=1))/back_up[fe_cols].mean(axis=1))
#         fe_data['rr_volatility_type'] = fe_data.apply(lambda x: 'Low' if x['rr_cov'] <= 0.4 else 'High', axis = 1)
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1)
        fe_data['rr_mean_intent_norm'] =  back_up[norm_intent].mean(axis=1)
        fe_data['rr_mean_intent_type'] = fe_data['rr_mean_intent'].apply(lambda x: 'Low' if x <= 3 else 'High')
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))
#         fe_data['rr_potential'] = fe_data['rr_need'] - fe_data['rr_mean_intent']
#         fe_data['rr_potential_per'] = (fe_data['rr_need'] - fe_data['rr_mean_intent'])/fe_data['rr_mean_intent']
#         fe_data['rr_potential_per_type'] = fe_data['rr_potential_per'].apply(lambda x: 'High' if x >= 0.5 else 'Low')
        fe_data['rr_trend_n'] = (  ((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis=1)) * (np.array(fe_data[fe_cols])*rr_weight).sum(axis=1))  - (np.array(fe_data[fe_cols]).sum(axis=1) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis=1))  ) 
        fe_data['rr_trend_d'] = ( ((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis = 1)) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[3],axis =1)) -(fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis =1))**2 )
        fe_data['rr_trend'] = fe_data['rr_trend_n']/fe_data['rr_trend_d']
        fe_data['rr_trend'] = fe_data['rr_trend'].fillna(0)
        fe_data['rr_intent_trend_type'] = fe_data['rr_trend'].apply(lambda x: intent_trend_type(x))
        fe=fe_data[fe_cols]
        fe=fe.ge(1.0).astype(int)
        fe_data['rr_regularity']= (1/91)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        fe_data['rr_regularity_type'] = fe_data['rr_regularity_scaled'].apply(lambda x: 'Low' if x <= 0.3 else 'High')
        fe_data['rr_recency_type'] = fe_data['days_since_last_rr'].apply(lambda x: recency_type(x))
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1)
        fe_data['rr_regularity_final_old'] = fe_data.apply(lambda x: regularity_decider(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        print('cp2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent']=fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        print('cp3')
#         fe_data['consideration_final'] = fe_data.apply(lambda x: consideration_signal_creator(x['rr_regularity_scaled'],x['rr_mean_intent'],x['active_days_mean']),axis=1)
#         fe_data['rr_intent_trend_type_v2']= fe_data.apply(lambda x: custom_func(x['rr_mean_intent_type'],x['rr_intent_trend_type']),axis=1)

        return(fe_data)

    

    print('here')
    L3_signals = create_L3_signals(fe_data,back_up)
            
            

    L3_signals['city']=city_name
    Final_data=L3_signals.copy()
    
#     Final_data = Final_data.add_suffix('_nov22')
    # Final_data['target']=np.where(Final_data.next_lts=='DORMANT',1,0)
    return Final_data

    

In [4]:
date_list=['2024-05-05','2024-05-12','2024-05-19' ]
# ,'2024-05-26','2024-06-02','2024-06-09','2024-06-16','2024-06-23']

# date_list=['2023-11-04','2023-11-11','2023-11-18']


In [5]:
base_week=date_list[0]
city='Bangalore'

In [6]:
concatenated_df=pd.DataFrame()
for i in date_list:
    print(i)
    tdf=consideration_query(i,city,base_week)
    print(tdf.shape[0])
    # concatenated_df=concatenated_df.append(tdf).reset_index(drop=True)
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-05-05
(698188, 93)
here
cp1
cp2
cp3
669005
2024-05-12
(698188, 93)
here
cp1
cp2
cp3
674887
2024-05-19
(698188, 93)
here
cp1
cp2
cp3
674949


In [7]:
concatenated_df['reg_intent']=concatenated_df.rr_regularity_final+'_'+concatenated_df.rr_mean_intent_final
base_df=concatenated_df[concatenated_df.run_date==base_week].reset_index(drop=True)

In [8]:
def stability_func(df,base_week):
    df_base_week=df[df.run_date==base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']].reset_index(drop=True)
    # ranking = {'rarely_1_rr':1,
    #            'rarely_2+_rr':2,
    #            'once_in_4_weeks_1_rr':3,
    #            'once_in_4_weeks_2+_rr':4,
    #         'once_in_2_weeks_1_rr':5,
    #            'once_in_2_weeks_2+_rr':6,
    #            'weekly_1_rr':7,
    #            'weekly_2+_rr':8}

    ranking = {
               '05_rare_need':1,
            '04_monthly':2,
               '03_bi_weekly':3,
               '02_weekly':4,
               '01_daily':5}

    intent_ranking = {
               '03_Low':1,
            '02_Medium':2,
               '01_High':3}


    final_rank={
    '05_rare_need_03_Low' : 1,
    '05_rare_need_02_Medium' : 2,
    '05_rare_need_01_High' : 3,
    '04_monthly_03_Low' : 4,
    '04_monthly_02_Medium' : 5,
    '04_monthly_01_High' : 6,
    '03_bi_weekly_03_Low' : 7,
    '03_bi_weekly_02_Medium' : 8,
    '03_bi_weekly_01_High' : 9,
    '02_weekly_03_Low' : 10,
    '02_weekly_02_Medium' : 11,
    '02_weekly_01_High' : 12,
    '01_daily_02_Medium' : 13,
    '01_daily_01_High' : 14
    }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)
    
    ## Regularity Stabillity

    pvt_df=pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df.columns


    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())




    dropped_reg.head()

    first_dropped_week = dropped_reg.idxmax(axis=1)


    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')




    pvt_df.reset_index(inplace=True)

    pvt_df.head()

    df_base_week_stability=pvt_df.merge(df_base_week,on=['customer'],how='inner')

    reg_stab=df_base_week_stability.groupby(by=['run_date','rr_regularity_final','first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_regularity_final',], columns='first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_regularity_final'],as_index=False).agg(
    base=('customer','nunique')),on=['rr_regularity_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    ## Intent Stabillity

    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df_int.columns


    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())




    dropped_int.head()

    int_first_dropped_week = dropped_int.idxmax(axis=1)


    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')




    pvt_df_int.reset_index(inplace=True)

    pvt_df_int.head()

    df_base_week_int_stability=pvt_df_int.merge(df_base_week,on=['customer'],how='inner')

    int_stab=df_base_week_int_stability.groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_mean_intent_final',], columns='int_first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_mean_intent_final'],as_index=False).agg(base=('customer','nunique')),on=['rr_mean_intent_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    ## Final Stabillity

    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df_final.columns


    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    # Display the result
    print(dropped_final.head())




    dropped_final.head()

    final_first_dropped_week = dropped_final.idxmax(axis=1)


    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')




    pvt_df_final.reset_index(inplace=True)

    pvt_df_final.head()

    df_base_week_final_stability=pvt_df_final.merge(df_base_week,on=['customer'],how='inner')

    final_stab=df_base_week_final_stability.groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_regularity_final','rr_mean_intent_final'], columns='final_first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_regularity_final','rr_mean_intent_final'],as_index=False).agg(base=('customer','nunique')),on=['rr_regularity_final','rr_mean_intent_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/final_stability.csv',index=False)

    return reg_stab,int_stab,final_stab



In [9]:
reg_df,int_df,final_seg_df=stability_func(concatenated_df,base_week)

run_date  2024-05-05  2024-05-12  2024-05-19
0              False        True        True
1              False       False       False
2              False       False       False
3              False       False       False
4              False       False       False
run_date  2024-05-05  2024-05-12  2024-05-19
0              False       False       False
1              False       False       False
2              False       False       False
3              False       False       False
4              False       False       False
run_date  2024-05-05  2024-05-12  2024-05-19
0              False        True        True
1              False       False       False
2              False       False       False
3              False       False       False
4              False       False       False


In [10]:
# complete flow
def stab_func_2(df,base_week,base_df,cons_type):
    
#     base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1=base_df.groupby(by=['city','run_date',cons_type],as_index=False).agg(base=('customer','nunique'))
    j=0
    for i in date_list:
        j+=1
        if i!=base_week:
            tdf=df[df.run_date==base_week][['run_date','customer',cons_type]].merge(df[df.run_date==i][['customer',cons_type]],on=['customer'],how='left').pivot_table(values='customer', index=['run_date',cons_type+'_x'], columns=cons_type+'_y', aggfunc='nunique').reset_index().fillna(0)
#             print(list(tdf))
            items_to_remove=['run_date',cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns=['run_date',cons_type]+[j+'_'+i for j in filtered_list]
            fdf1=fdf1.merge(tdf,on=['run_date',cons_type],how='inner')
    return fdf1
        
        

In [11]:
#cons_type: 'rr_regularity_final','rr_mean_intent_final','reg_intent'
finaldf=stab_func_2(concatenated_df,base_week,base_df,'reg_intent')

In [12]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-05-12,01_daily_02_Medium_2024-05-12,02_weekly_01_High_2024-05-12,02_weekly_02_Medium_2024-05-12,02_weekly_03_Low_2024-05-12,03_bi_weekly_01_High_2024-05-12,03_bi_weekly_02_Medium_2024-05-12,03_bi_weekly_03_Low_2024-05-12,04_monthly_01_High_2024-05-12,04_monthly_02_Medium_2024-05-12,04_monthly_03_Low_2024-05-12,05_rare_need_01_High_2024-05-12,05_rare_need_02_Medium_2024-05-12,05_rare_need_03_Low_2024-05-12,01_daily_01_High_2024-05-19,01_daily_02_Medium_2024-05-19,02_weekly_01_High_2024-05-19,02_weekly_02_Medium_2024-05-19,02_weekly_03_Low_2024-05-19,03_bi_weekly_01_High_2024-05-19,03_bi_weekly_02_Medium_2024-05-19,03_bi_weekly_03_Low_2024-05-19,04_monthly_01_High_2024-05-19,04_monthly_02_Medium_2024-05-19,04_monthly_03_Low_2024-05-19,05_rare_need_01_High_2024-05-19,05_rare_need_02_Medium_2024-05-19,05_rare_need_03_Low_2024-05-19
0,Bangalore,2024-05-05,01_daily_01_High,77353,71944.0,2390.0,779.0,610.0,0.0,1584.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68485.0,3365.0,859.0,1144.0,0.0,2692.0,406.0,0.0,402.0,0.0,0.0,0.0,0.0,0.0
1,Bangalore,2024-05-05,01_daily_02_Medium,31749,3678.0,23632.0,19.0,3226.0,10.0,39.0,1145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5139.0,19731.0,19.0,4544.0,13.0,167.0,1939.0,11.0,0.0,186.0,0.0,0.0,0.0,0.0
2,Bangalore,2024-05-05,02_weekly_01_High,5361,910.0,18.0,2898.0,854.0,0.0,665.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1119.0,28.0,2176.0,1028.0,1.0,577.0,187.0,0.0,245.0,0.0,0.0,0.0,0.0,0.0
3,Bangalore,2024-05-05,02_weekly_02_Medium,119295,1037.0,4522.0,1063.0,97367.0,2194.0,25.0,12811.0,276.0,0.0,0.0,0.0,0.0,0.0,0.0,2098.0,6198.0,1292.0,85946.0,3008.0,201.0,18525.0,1084.0,0.0,943.0,0.0,0.0,0.0,0.0
4,Bangalore,2024-05-05,02_weekly_03_Low,34887,52.0,44.0,31.0,4477.0,20344.0,0.0,216.0,9723.0,0.0,0.0,0.0,0.0,0.0,0.0,64.0,62.0,12.0,5487.0,15590.0,27.0,1142.0,9589.0,0.0,0.0,2914.0,0.0,0.0,0.0
5,Bangalore,2024-05-05,03_bi_weekly_01_High,10432,2197.0,149.0,329.0,169.0,0.0,6367.0,867.0,1.0,328.0,25.0,0.0,0.0,0.0,0.0,3434.0,292.0,393.0,381.0,0.0,4230.0,1114.0,9.0,517.0,58.0,4.0,0.0,0.0,0.0
6,Bangalore,2024-05-05,03_bi_weekly_02_Medium,99272,381.0,1678.0,140.0,15058.0,697.0,1329.0,69543.0,4043.0,27.0,6057.0,319.0,0.0,0.0,0.0,1060.0,2591.0,287.0,19846.0,1361.0,1355.0,56821.0,6132.0,80.0,8912.0,827.0,0.0,0.0,0.0
7,Bangalore,2024-05-05,03_bi_weekly_03_Low,94358,0.0,5.0,0.0,1470.0,7335.0,6.0,6806.0,66500.0,0.0,271.0,11965.0,0.0,0.0,0.0,4.0,32.0,4.0,3473.0,8639.0,19.0,8592.0,56122.0,0.0,604.0,16869.0,0.0,0.0,0.0
8,Bangalore,2024-05-05,04_monthly_01_High,4753,0.0,0.0,0.0,0.0,0.0,1514.0,342.0,0.0,2345.0,412.0,4.0,121.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,2079.0,679.0,5.0,1328.0,397.0,14.0,196.0,49.0,6.0
9,Bangalore,2024-05-05,04_monthly_02_Medium,41608,0.0,0.0,0.0,0.0,0.0,463.0,11915.0,1455.0,503.0,23705.0,1918.0,7.0,1521.0,121.0,0.0,0.0,0.0,0.0,0.0,992.0,13534.0,2280.0,273.0,18377.0,2957.0,36.0,2744.0,415.0


In [13]:
base_df.groupby(by=['city_name','run_date','reg_intent'],as_index=False).agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    
    )
# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Bangalore,2024-05-05,01_daily_01_High,77353,71136,73885,74792,75266,75522,75709,75845,75922,76056,76056,70021,73191,74283,74862,75170,75397,75623,75755,75874,75874,65797,71165,73000,73867,74416,74767,75075,75257,75414,75414
1,Bangalore,2024-05-05,01_daily_02_Medium,31749,28348,30059,30548,30807,30944,31029,31101,31131,31183,31183,27449,29582,30247,30584,30762,30875,30983,31046,31094,31094,24182,27795,29135,29725,30125,30350,30549,30670,30758,30758
2,Bangalore,2024-05-05,02_weekly_01_High,5361,4222,4646,4799,4874,4922,4948,4968,4981,4988,4988,4024,4533,4708,4796,4856,4885,4909,4929,4935,4935,3627,4262,4496,4621,4718,4775,4812,4845,4859,4859
3,Bangalore,2024-05-05,02_weekly_02_Medium,119295,95490,107772,111720,113638,114657,115269,115669,115901,116159,116159,89418,103954,109142,111871,113282,114157,114760,115166,115450,115450,75202,94197,102322,106669,109368,111063,112243,113044,113603,113603
4,Bangalore,2024-05-05,02_weekly_03_Low,34887,22520,27376,29224,30229,30822,31174,31372,31508,31656,31656,19895,25231,27510,28939,29723,30242,30574,30806,30988,30988,15779,21498,24578,26390,27631,28484,29073,29487,29812,29812
5,Bangalore,2024-05-05,03_bi_weekly_01_High,10432,8505,9265,9579,9785,9905,9976,10018,10051,10083,10083,8156,9001,9364,9602,9740,9842,9917,9973,10010,10010,7450,8548,9016,9319,9515,9668,9771,9834,9888,9888
6,Bangalore,2024-05-05,03_bi_weekly_02_Medium,99272,68938,82846,88454,91698,93555,94599,95259,95651,96016,96016,61889,77266,84166,88371,90908,92448,93494,94190,94695,94695,50236,67074,76114,81481,85223,87726,89484,90714,91606,91606
7,Bangalore,2024-05-05,03_bi_weekly_03_Low,94358,54655,71336,78931,83616,86382,88085,89066,89733,90246,90246,45920,62953,71782,77873,81548,84046,85714,86904,87668,87668,34887,51370,61452,68256,73097,76733,79350,81304,82652,82652
8,Bangalore,2024-05-05,04_monthly_01_High,4753,3395,3752,3945,4065,4144,4204,4248,4275,4304,4304,3187,3593,3807,3947,4037,4105,4152,4199,4231,4231,2877,3371,3616,3776,3893,3978,4048,4106,4142,4142
9,Bangalore,2024-05-05,04_monthly_02_Medium,41608,23488,29494,32479,34552,35854,36729,37313,37694,38073,38073,20220,26297,29545,32030,33635,34802,35635,36223,36727,36727,16082,22080,25770,28400,30364,31904,33038,33899,34600,34600


In [14]:
test=list(finaldf)

In [15]:
# test.remove([['city','01_daily_2023-11-11']])

In [16]:
test=['city']+[i+'_yo' for i in test]

In [17]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-05-12_yo',
 '01_daily_02_Medium_2024-05-12_yo',
 '02_weekly_01_High_2024-05-12_yo',
 '02_weekly_02_Medium_2024-05-12_yo',
 '02_weekly_03_Low_2024-05-12_yo',
 '03_bi_weekly_01_High_2024-05-12_yo',
 '03_bi_weekly_02_Medium_2024-05-12_yo',
 '03_bi_weekly_03_Low_2024-05-12_yo',
 '04_monthly_01_High_2024-05-12_yo',
 '04_monthly_02_Medium_2024-05-12_yo',
 '04_monthly_03_Low_2024-05-12_yo',
 '05_rare_need_01_High_2024-05-12_yo',
 '05_rare_need_02_Medium_2024-05-12_yo',
 '05_rare_need_03_Low_2024-05-12_yo',
 '01_daily_01_High_2024-05-19_yo',
 '01_daily_02_Medium_2024-05-19_yo',
 '02_weekly_01_High_2024-05-19_yo',
 '02_weekly_02_Medium_2024-05-19_yo',
 '02_weekly_03_Low_2024-05-19_yo',
 '03_bi_weekly_01_High_2024-05-19_yo',
 '03_bi_weekly_02_Medium_2024-05-19_yo',
 '03_bi_weekly_03_Low_2024-05-19_yo',
 '04_monthly_01_High_2024-05-19_yo',
 '04_monthly_02_Medium_2024-05-19_yo',
 '04_monthly_03_Low_2024-05-1

In [18]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-05-12_yo',
 '01_daily_02_Medium_2024-05-12_yo',
 '02_weekly_01_High_2024-05-12_yo',
 '02_weekly_02_Medium_2024-05-12_yo',
 '02_weekly_03_Low_2024-05-12_yo',
 '03_bi_weekly_01_High_2024-05-12_yo',
 '03_bi_weekly_02_Medium_2024-05-12_yo',
 '03_bi_weekly_03_Low_2024-05-12_yo',
 '04_monthly_01_High_2024-05-12_yo',
 '04_monthly_02_Medium_2024-05-12_yo',
 '04_monthly_03_Low_2024-05-12_yo',
 '05_rare_need_01_High_2024-05-12_yo',
 '05_rare_need_02_Medium_2024-05-12_yo',
 '05_rare_need_03_Low_2024-05-12_yo',
 '01_daily_01_High_2024-05-19_yo',
 '01_daily_02_Medium_2024-05-19_yo',
 '02_weekly_01_High_2024-05-19_yo',
 '02_weekly_02_Medium_2024-05-19_yo',
 '02_weekly_03_Low_2024-05-19_yo',
 '03_bi_weekly_01_High_2024-05-19_yo',
 '03_bi_weekly_02_Medium_2024-05-19_yo',
 '03_bi_weekly_03_Low_2024-05-19_yo',
 '04_monthly_01_High_2024-05-19_yo',
 '04_monthly_02_Medium_2024-05-19_yo',
 '04_monthly_03_Low_2024-05-1

In [19]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-05-12,01_daily_02_Medium_2024-05-12,02_weekly_01_High_2024-05-12,02_weekly_02_Medium_2024-05-12,02_weekly_03_Low_2024-05-12,03_bi_weekly_01_High_2024-05-12,03_bi_weekly_02_Medium_2024-05-12,03_bi_weekly_03_Low_2024-05-12,04_monthly_01_High_2024-05-12,04_monthly_02_Medium_2024-05-12,04_monthly_03_Low_2024-05-12,05_rare_need_01_High_2024-05-12,05_rare_need_02_Medium_2024-05-12,05_rare_need_03_Low_2024-05-12,01_daily_01_High_2024-05-19,01_daily_02_Medium_2024-05-19,02_weekly_01_High_2024-05-19,02_weekly_02_Medium_2024-05-19,02_weekly_03_Low_2024-05-19,03_bi_weekly_01_High_2024-05-19,03_bi_weekly_02_Medium_2024-05-19,03_bi_weekly_03_Low_2024-05-19,04_monthly_01_High_2024-05-19,04_monthly_02_Medium_2024-05-19,04_monthly_03_Low_2024-05-19,05_rare_need_01_High_2024-05-19,05_rare_need_02_Medium_2024-05-19,05_rare_need_03_Low_2024-05-19
0,Bangalore,2024-05-05,01_daily_01_High,77353,71944.0,2390.0,779.0,610.0,0.0,1584.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68485.0,3365.0,859.0,1144.0,0.0,2692.0,406.0,0.0,402.0,0.0,0.0,0.0,0.0,0.0
1,Bangalore,2024-05-05,01_daily_02_Medium,31749,3678.0,23632.0,19.0,3226.0,10.0,39.0,1145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5139.0,19731.0,19.0,4544.0,13.0,167.0,1939.0,11.0,0.0,186.0,0.0,0.0,0.0,0.0
2,Bangalore,2024-05-05,02_weekly_01_High,5361,910.0,18.0,2898.0,854.0,0.0,665.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1119.0,28.0,2176.0,1028.0,1.0,577.0,187.0,0.0,245.0,0.0,0.0,0.0,0.0,0.0
3,Bangalore,2024-05-05,02_weekly_02_Medium,119295,1037.0,4522.0,1063.0,97367.0,2194.0,25.0,12811.0,276.0,0.0,0.0,0.0,0.0,0.0,0.0,2098.0,6198.0,1292.0,85946.0,3008.0,201.0,18525.0,1084.0,0.0,943.0,0.0,0.0,0.0,0.0
4,Bangalore,2024-05-05,02_weekly_03_Low,34887,52.0,44.0,31.0,4477.0,20344.0,0.0,216.0,9723.0,0.0,0.0,0.0,0.0,0.0,0.0,64.0,62.0,12.0,5487.0,15590.0,27.0,1142.0,9589.0,0.0,0.0,2914.0,0.0,0.0,0.0
5,Bangalore,2024-05-05,03_bi_weekly_01_High,10432,2197.0,149.0,329.0,169.0,0.0,6367.0,867.0,1.0,328.0,25.0,0.0,0.0,0.0,0.0,3434.0,292.0,393.0,381.0,0.0,4230.0,1114.0,9.0,517.0,58.0,4.0,0.0,0.0,0.0
6,Bangalore,2024-05-05,03_bi_weekly_02_Medium,99272,381.0,1678.0,140.0,15058.0,697.0,1329.0,69543.0,4043.0,27.0,6057.0,319.0,0.0,0.0,0.0,1060.0,2591.0,287.0,19846.0,1361.0,1355.0,56821.0,6132.0,80.0,8912.0,827.0,0.0,0.0,0.0
7,Bangalore,2024-05-05,03_bi_weekly_03_Low,94358,0.0,5.0,0.0,1470.0,7335.0,6.0,6806.0,66500.0,0.0,271.0,11965.0,0.0,0.0,0.0,4.0,32.0,4.0,3473.0,8639.0,19.0,8592.0,56122.0,0.0,604.0,16869.0,0.0,0.0,0.0
8,Bangalore,2024-05-05,04_monthly_01_High,4753,0.0,0.0,0.0,0.0,0.0,1514.0,342.0,0.0,2345.0,412.0,4.0,121.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,2079.0,679.0,5.0,1328.0,397.0,14.0,196.0,49.0,6.0
9,Bangalore,2024-05-05,04_monthly_02_Medium,41608,0.0,0.0,0.0,0.0,0.0,463.0,11915.0,1455.0,503.0,23705.0,1918.0,7.0,1521.0,121.0,0.0,0.0,0.0,0.0,0.0,992.0,13534.0,2280.0,273.0,18377.0,2957.0,36.0,2744.0,415.0
